In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, year, month, dayofmonth, hour

def main():
    spark = SparkSession.builder \
        .appName("bronze_writer_prices") \
        .getOrCreate()

    df = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "kafka:9092") \
        .option("subscribe", "crypto.prices") \
        .option("startingOffsets", "latest") \
        .option("failOnDataLoss", "false") \
        .load()

    bronze = df.select(
        col("value").cast("string").alias("kafka_value_raw"),
        col("key").cast("string").alias("kafka_key"),
        col("topic").alias("kafka_topic"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp"),
        current_timestamp().alias("ingestion_ts"),
    ).withColumn("year", year("ingestion_ts")) \
     .withColumn("month", month("ingestion_ts")) \
     .withColumn("day", dayofmonth("ingestion_ts")) \
     .withColumn("hour", hour("ingestion_ts"))

    query = bronze.writeStream \
        .format("parquet") \
        .option("path", "s3a://bucket/bronze/prices/") \
        .option("checkpointLocation", "s3a://bucket/checkpoints/bronze_prices/") \
        .partitionBy("year", "month", "day", "hour") \
        .outputMode("append") \
        .trigger(processingTime="30 seconds") \
        .start()

    query.awaitTermination()

if __name__ == "__main__":
    main()

Py4JJavaError: An error occurred while calling o74.start.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.streaming.FileStreamSink.<init>(FileStreamSink.scala:135)
	at org.apache.spark.sql.execution.datasources.DataSource.createSink(DataSource.scala:322)
	at org.apache.spark.sql.streaming.DataStreamWriter.createV1Sink(DataStreamWrite

In [4]:
SparkSession.stop()

TypeError: SparkSession.stop() missing 1 required positional argument: 'self'